# 03 — Training Loop from Scratch

This notebook shows the full training cycle in plain PyTorch:
`forward -> loss -> backward -> optimizer step -> zero_grad`.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 1) Create synthetic classification data

Two Gaussian blobs in 2D. Classify each point as 0 or 1.

In [ ]:
n = 1000
x0 = torch.randn(n // 2, 2) * 0.7 + torch.tensor([-1.5, -1.0])
x1 = torch.randn(n // 2, 2) * 0.8 + torch.tensor([1.5, 1.0])
X = torch.cat([x0, x1], dim=0)
y = torch.cat([torch.zeros(n // 2), torch.ones(n // 2)]).long()

dataset = TensorDataset(X, y)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128)

print("Train samples:", len(train_ds), "Val samples:", len(val_ds))

## 2) Define model, loss, optimizer

In [ ]:
model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 2)
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

print(model)

## 3) Training & validation loops

In [ ]:
def run_epoch(loader, training=True):
    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        if training:
            optimizer.zero_grad()

        logits = model(xb)
        loss = criterion(logits, yb)

        if training:
            loss.backward()
            optimizer.step()

        preds = logits.argmax(dim=1)
        total_correct += (preds == yb).sum().item()
        total_samples += yb.size(0)
        total_loss += loss.item() * yb.size(0)

    avg_loss = total_loss / total_samples
    avg_acc = total_correct / total_samples
    return avg_loss, avg_acc

In [ ]:
epochs = 20
history = []

for epoch in range(1, epochs + 1):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    with torch.no_grad():
        val_loss, val_acc = run_epoch(val_loader, training=False)

    history.append((train_loss, train_acc, val_loss, val_acc))

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:>2} | "
            f"train_loss={train_loss:.4f}, train_acc={train_acc:.3f} | "
            f"val_loss={val_loss:.4f}, val_acc={val_acc:.3f}"
        )

## 4) Quick sanity checks

In [ ]:
final_train_loss, final_train_acc, final_val_loss, final_val_acc = history[-1]
print("Final train acc:", round(final_train_acc, 4))
print("Final val acc:", round(final_val_acc, 4))

assert final_train_acc > 0.9, "Train accuracy is lower than expected."
assert final_val_acc > 0.9, "Validation accuracy is lower than expected."
print("Sanity checks passed ✅")

## 5) Exercises

1. Replace `Adam` with `SGD(momentum=0.9)` and compare speed.
2. Increase model width from 16 to 64 and observe overfitting signs.
3. Add an early-stopping condition when validation loss stops improving.

Next notebook: `04_your_first_nn.ipynb`.